# Facebook Denoiser

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [2]:
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-Denoiser')

control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

## Load Model

In [3]:
from denoiser import pretrained

model_name = 'dns64'
model = pretrained.dns64()

# 设备检测
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

model = model.to(device)
model.eval()

target_sr = model.sample_rate  # denoiser 模型自带采样率（通常 16000）
print(f"Model: {model_name}, Sample Rate: {target_sr}, Device: {device}")

Downloading: "https://dl.fbaipublicfiles.com/adiyoss/denoiser/dns64-a7761ff99a7d5bb6.th" to /Users/sleepwalker/.cache/torch/hub/checkpoints/dns64-a7761ff99a7d5bb6.th


100%|██████████| 128M/128M [00:04<00:00, 28.7MB/s] 


Model: dns64, Sample Rate: 16000, Device: mps


## Denoise Function

In [4]:
def denoise_audio(audio_path, model, device, target_sr):
    """
    使用 Facebook Denoiser 进行语音降噪
    """
    audio, sr = sf.read(str(audio_path))

    # 多声道转单声道
    if len(audio.shape) == 2:
        audio = np.mean(audio, axis=1)

    # 重采样到目标采样率
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)

    audio = audio.astype(np.float32)

    # 归一化到 [-1, 1]
    audio = np.clip(audio, -1.0, 1.0)

    # 转为 torch tensor: [batch, channels, time]
    wav = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        denoised = model(wav)

    # 转回 numpy: [time]
    denoised_audio = denoised.squeeze().cpu().numpy()
    denoised_audio = np.clip(denoised_audio, -1.0, 1.0)

    return denoised_audio, target_sr

In [ ]:
def batch_denoise(files, output_subdir, model, device, target_sr, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）

    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: denoiser 模型实例
        device: 计算设备
        target_sr: 目标采样率
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        output_file = output_subdir / audio_file.name

        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue

        try:
            clear_memory()

            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, model, device, target_sr)

            # 保存（16位整数格式）
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1

            # ⚡ 处理后立即清理显存
            del denoised_audio  # 删除大数组
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()

    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Execute denoise function

In [ ]:
clear_memory()

batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    model,
    device,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files,
    output_dir / 'Control',
    model,
    device,
    target_sr=target_sr,
    group_name='Control'
)

Processing Dementia:  87%|████████▋ | 268/309 [06:57<01:09,  1.69s/it]